# 18wD — Locked holdout and June probability evaluation

This stage first verifies that the 18wC probability release is outcome blind and that all calibration parameters were frozen in development. It then unblinds the certified HKO outcomes and evaluates uncalibrated and calibrated contract probabilities.

Two supports are reported:

- **model-only support:** all 40 internal-holdout and 119 June weather-ready books per candidate and probability variant;
- **exact market-common support:** 40 internal-holdout and 114 June books for which the full eleven-contract market book is also available.

Model books are coherent by construction. Market categorical scores use probabilities normalised within each complete book; raw market probabilities are retained for binary diagnostics and later trading analysis. Market information is never used for model fitting, candidate selection or calibration.

**Revision v2.** The notebook verifies the 18wA-C source inventories and the installed outcome-free contract-definition artefact before unblinding. Figures now use concise labels, separate holdout and June panels, and restricted axes with annotated CatBoost outliers.

In [1]:
from __future__ import annotations
import hashlib, json, math, platform, sys
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT=Path.cwd().resolve()
if not (ROOT/'.git').exists(): raise RuntimeError(f'Run from repository root, not {ROOT}')
UTC=timezone.utc; STEP='18wD'; EPS=1e-12; NC=11
S=ROOT/'data/processed/18s_expanded_march_june_canonical_sample'; WA=ROOT/'data/processed/18wA_contract_probability_mapping'; WB=ROOT/'data/processed/18wB_development_probability_calibration'; WC=ROOT/'data/processed/18wC_blind_calibrated_probability_release'; DEF=ROOT/'data/manual/18w_outcome_free_contract_definitions'
CONTRACT=S/'18s_expanded_certified_contract_outcome_panel.csv'; MARKET=S/'18s_expanded_market_scoring_panel.csv'; COMMON=S/'18s_expanded_exact_common_support_panel.csv'; S_MAN=S/'18s_expanded_sha256_manifest.csv'; S_SUM=S/'18s_expanded_sample_summary.json'
RELEASE=WC/'18wC_blind_probability_release.csv'; WC_MAN=WC/'18wC_sha256_manifest.csv'; WC_SUM=WC/'18wC_summary.json'; WC_PROTOCOL=WC/'18wC_protocol.json'; WC_SOURCES=WC/'18wC_source_inventory.csv'
PARAM=WB/'18wB_selected_calibration_parameters.csv'; WB_MAN=WB/'18wB_sha256_manifest.csv'; WB_SUM=WB/'18wB_summary.json'; WB_PROTOCOL=WB/'18wB_protocol.json'; WB_SOURCES=WB/'18wB_source_inventory.csv'
WA_MAN=WA/'18wA_sha256_manifest.csv'; WA_SUM=WA/'18wA_summary.json'; WA_PROTOCOL=WA/'18wA_protocol.json'; WA_SOURCES=WA/'18wA_source_inventory.csv'
DEF_PANEL=DEF/'18w_outcome_free_contract_definition_panel.csv'; DEF_META=DEF/'18w_outcome_free_contract_definition_metadata.json'
OUT=ROOT/'data/processed/18wD_holdout_external_probability_evaluation'; REPORT=ROOT/'reports/18wD_holdout_external_probability_evaluation'; FIG=REPORT/'figures'; OUT.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)
for p in [CONTRACT,MARKET,COMMON,S_MAN,S_SUM,RELEASE,WC_MAN,WC_SUM,WC_PROTOCOL,WC_SOURCES,PARAM,WB_MAN,WB_SUM,WB_PROTOCOL,WB_SOURCES,WA_MAN,WA_SUM,WA_PROTOCOL,WA_SOURCES,DEF_PANEL,DEF_META]:
    if not p.is_file(): raise FileNotFoundError(p)

def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for c in iter(lambda:f.read(1024*1024),b''): h.update(c)
    return h.hexdigest()
def verify(p):
    bad=[]
    for r in pd.read_csv(p).itertuples(index=False):
        q=ROOT/r.path
        if not q.is_file(): bad.append('MISSING '+r.path); continue
        if sha(q)!=r.sha256: bad.append('HASH '+r.path)
        if q.stat().st_size!=int(r.size_bytes): bad.append('SIZE '+r.path)
    if bad: raise AssertionError('\n'.join(bad))
def pbool(s,name):
    if pd.api.types.is_bool_dtype(s): return s.astype(bool)
    x=s.astype(str).str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False,'yes':True,'no':False})
    if x.isna().any(): raise ValueError(f'Cannot parse {name}: {s[x.isna()].drop_duplicates().tolist()}')
    return x.astype(bool)
def date_weights(df,date_col='event_date'):
    n=df[date_col].nunique(); w=1/(n*df.groupby(date_col)[date_col].transform('size'))
    if not np.isclose(w.sum(),1,atol=1e-12): raise AssertionError('Date-balanced weights')
    return w
def binary_log(p,y):
    p=np.clip(np.asarray(p,float),EPS,1-EPS); y=np.asarray(y,float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))

SHORT_NAME={'pooled_empirical_residual':'Empirical','gp_matern32_rule':'Matérn GP','catboost_quantile_pooled':'CatBoost'}
SHORT_VARIANT={'UNCALIBRATED':'Uncal.','CALIBRATED':'Cal.'}
BLOCK_TITLE={'INTERNAL_HOLDOUT':'Internal holdout','EXTERNAL_TEST':'June external test'}
def save_horizontal(frame,label_col,value_col,title,xlabel,path,xlim=None,clip_annotations=False,zero_line=False):
    z=frame.reset_index(drop=True).copy(); values=z[value_col].to_numpy(float); plotted=values.copy()
    if xlim is not None:
        plotted=np.clip(plotted,xlim[0],xlim[1])
    y=np.arange(len(z)); fig,ax=plt.subplots(figsize=(8.6,max(4.2,0.55*len(z)+1.4))); ax.barh(y,plotted); ax.set_yticks(y); ax.set_yticklabels(z[label_col]); ax.invert_yaxis(); ax.set_title(title); ax.set_xlabel(xlabel)
    if xlim is not None: ax.set_xlim(*xlim)
    if zero_line: ax.axvline(0,linewidth=0.8)
    for i,(actual,shown) in enumerate(zip(values,plotted)):
        clipped=not np.isclose(actual,shown,rtol=0,atol=1e-12)
        if clipped and clip_annotations:
            span=xlim[1]-xlim[0]
            anchor=(xlim[1]-0.03*span) if actual>xlim[1] else (xlim[0]+0.03*span)
            ha='right' if actual>xlim[1] else 'left'
            ax.text(anchor,i,f'actual {actual:.2f}',va='center',ha=ha,fontsize=8)
        else:
            offset=0.02*(xlim[1]-xlim[0]) if xlim is not None else 0.02*max(abs(values).max(),1)
            ax.text(shown+(offset if shown>=0 else -offset),i,f'{actual:.3f}',va='center',ha='left' if shown>=0 else 'right',fontsize=8)
    fig.tight_layout(); fig.savefig(path,dpi=220,bbox_inches='tight'); plt.close(fig)

# Verify the mechanically outcome-blind A-C chain before loading outcomes or market data.
for p in [WA_MAN,WB_MAN,WC_MAN]: verify(p)
wa_summary=json.loads(WA_SUM.read_text()); wb_summary=json.loads(WB_SUM.read_text()); wc_summary=json.loads(WC_SUM.read_text())
wa_protocol=json.loads(WA_PROTOCOL.read_text()); wb_protocol=json.loads(WB_PROTOCOL.read_text()); wc_protocol=json.loads(WC_PROTOCOL.read_text())
for stage,summary in [('18wA',wa_summary),('18wB',wb_summary),('18wC',wc_summary)]:
    if summary.get('verdict')!='PASS': raise AssertionError(f'{stage} is not PASS')
for stage,protocol in [('18wA',wa_protocol),('18wB',wb_protocol),('18wC',wc_protocol)]:
    if protocol.get('full_outcome_panel_loaded') is not False: raise AssertionError(f'{stage} full-outcome boundary failed')
    if protocol.get('market_information_used') is not False: raise AssertionError(f'{stage} market boundary failed')
if wc_protocol.get('holdout_or_external_outcomes_loaded') is not False or wc_protocol.get('refit_on_holdout_labels') is not False: raise AssertionError('Blind protocol boundary failed')
forbidden_pre_unblind_paths={'18s_expanded_certified_contract_outcome_panel.csv','18s_expanded_market_scoring_panel.csv','18s_expanded_exact_common_support_panel.csv'}
pre_unblind_sources=pd.concat([pd.read_csv(WA_SOURCES),pd.read_csv(WB_SOURCES),pd.read_csv(WC_SOURCES)],ignore_index=True)
if pre_unblind_sources.path.astype(str).map(lambda value:any(token in value for token in forbidden_pre_unblind_paths)).any(): raise AssertionError('A pre-unblinding source inventory references outcomes or market data')
def_meta=json.loads(DEF_META.read_text())
if sha(DEF_PANEL)!=def_meta.get('output_sha256') or def_meta.get('forbidden_outcome_columns_present') is not False: raise AssertionError('Outcome-free definition artefact boundary failed')
release=pd.read_csv(RELEASE,dtype={'market_id':str},low_memory=False); release['event_date']=pd.to_datetime(release.event_date,errors='raise'); release['outcome_blind']=pbool(release.outcome_blind,'outcome blind'); release['refit_on_holdout_labels']=pbool(release.refit_on_holdout_labels,'refit')
forbidden={'hko_daily_max_c','Y_event_int','Y_no_int','residual_c','forecast_error_c','p_market','market_binary_brier','market_binary_log_score','current_label_available_utc'}
if len(release)!=10494 or forbidden.intersection(release.columns) or not release.outcome_blind.all() or release.refit_on_holdout_labels.any(): raise AssertionError('18wC release is not blind')
params=pd.read_csv(PARAM)
if len(params)!=3: raise AssertionError('Calibration parameter rows')
print('Outcome-blind A-C chain and frozen calibration verified before unblinding: PASS')

# Unblind only after the preceding checks.
verify(S_MAN)
if json.loads(S_SUM.read_text()).get('verdict')!='PASS': raise AssertionError('18s not PASS')
contracts=pd.read_csv(CONTRACT,dtype={'market_id':str},low_memory=False); market=pd.read_csv(MARKET,dtype={'market_id':str},low_memory=False); common=pd.read_csv(COMMON,dtype={'market_id':str},low_memory=False)
for df in [contracts,market,common]: df['event_date']=pd.to_datetime(df.event_date,errors='raise')
definitions=pd.read_csv(DEF_PANEL,dtype={'market_id':str},low_memory=False); definitions['event_date']=pd.to_datetime(definitions.event_date,errors='raise')
definition_columns=def_meta['columns']
left=definitions[definition_columns].sort_values(['event_date','market_id']).reset_index(drop=True)
right=contracts[definition_columns].sort_values(['event_date','market_id']).reset_index(drop=True)
pd.testing.assert_frame_equal(left,right,check_dtype=False,check_like=False)
outcomes=contracts[['event_date','market_id','hko_daily_max_c','Y_event_int','Y_no_int','canonical_label','event_type','lower_bound_c','upper_bound_c']].copy()
model=release.merge(outcomes,on=['event_date','market_id'],how='left',validate='many_to_one',suffixes=('','_outcome'))
if len(model)!=10494 or model[['hko_daily_max_c','Y_event_int']].isna().any().any(): raise AssertionError('Outcome join failed')
if not model.groupby(['candidate_id','probability_variant','event_date','decision_rule']).Y_event_int.sum().eq(1).all(): raise AssertionError('Winner count after unblinding')
# Model-only book scores.
bookrows=[]
for k,g in model.groupby(['candidate_id','probability_variant','evaluation_block','event_date','decision_rule'],sort=True):
    p=g.p_model.to_numpy(float); y=g.Y_event_int.to_numpy(float); win=float(p[y==1][0]); winning=g[g.Y_event_int.eq(1)].iloc[0]
    bookrows.append({'candidate_id':k[0],'probability_variant':k[1],'evaluation_block':k[2],'event_date':k[3],'decision_rule':k[4],'decision_rule_order':int(g.decision_rule_order.iloc[0]),'contract_rows':len(g),'probability_sum':float(p.sum()),'winning_market_id':str(winning.market_id),'winning_label':str(winning.canonical_label),'winning_probability':win,'categorical_log_score':-math.log(min(max(win,EPS),1.0)),'multiclass_brier':float(np.square(p-y).sum()),'zero_winning_probability':win==0,'hko_daily_max_c':float(winning.hko_daily_max_c)})
books=pd.DataFrame(bookrows)
if len(books)!=954 or not np.isclose(books.probability_sum,1,atol=1e-12).all(): raise AssertionError('Model book scores')
summaryrows=[]
for k,g in books.groupby(['candidate_id','probability_variant','evaluation_block'],sort=True):
    w=date_weights(g)
    summaryrows.append({'candidate_id':k[0],'probability_variant':k[1],'evaluation_block':k[2],'books':len(g),'dates':g.event_date.nunique(),'date_balanced_mean_categorical_log_score':float(np.average(g.categorical_log_score,weights=w)),'date_balanced_mean_multiclass_brier':float(np.average(g.multiclass_brier,weights=w)),'date_balanced_mean_winning_probability':float(np.average(g.winning_probability,weights=w)),'zero_winning_probability_books':int(g.zero_winning_probability.sum())})
model_summary=pd.DataFrame(summaryrows)
if len(model_summary)!=12: raise AssertionError('Model summary rows')
# Exact complete market-common support.
eval_common=common[common.event_date.between(pd.Timestamp('2026-05-22'),pd.Timestamp('2026-06-30'))].copy(); eval_common['evaluation_block']=np.where(eval_common.event_date.le(pd.Timestamp('2026-05-31')),'INTERNAL_HOLDOUT','EXTERNAL_TEST')
if len(eval_common)!=1694 or eval_common[['event_date','decision_rule']].drop_duplicates().shape[0]!=154: raise AssertionError('Exact common support totals')
if not eval_common.groupby(['event_date','decision_rule']).size().eq(11).all() or not eval_common.groupby(['event_date','decision_rule']).Y_event_int.sum().eq(1).all(): raise AssertionError('Incomplete exact common books')
eval_common['market_book_sum_raw']=eval_common.groupby(['event_date','decision_rule']).p_market.transform('sum')
if (eval_common.market_book_sum_raw<=0).any(): raise AssertionError('Non-positive market book sum')
eval_common['p_market_normalised']=eval_common.p_market/eval_common.market_book_sum_raw
if not eval_common.groupby(['event_date','decision_rule']).p_market_normalised.sum().apply(lambda v:np.isclose(v,1,atol=1e-12)).all(): raise AssertionError('Market normalization')
keys=eval_common[['event_date','decision_rule','market_id','p_market','p_market_normalised','market_book_sum_raw','price_staleness_hours','selected_price_timestamp_utc']].copy()
exact=model.merge(keys,on=['event_date','decision_rule','market_id'],how='inner',validate='many_to_one')
if len(exact)!=10164: raise AssertionError(f'Exact joined rows {len(exact)}')
# Binary scores at contract level.
exact['model_binary_brier']=np.square(exact.p_model-exact.Y_event_int); exact['model_binary_log_score']=binary_log(exact.p_model,exact.Y_event_int); exact['market_raw_binary_brier']=np.square(exact.p_market-exact.Y_event_int); exact['market_raw_binary_log_score']=binary_log(exact.p_market,exact.Y_event_int); exact['market_normalised_binary_brier']=np.square(exact.p_market_normalised-exact.Y_event_int); exact['market_normalised_binary_log_score']=binary_log(exact.p_market_normalised,exact.Y_event_int)
# Book-level paired scores.
pairedrows=[]
for k,g in exact.groupby(['candidate_id','probability_variant','evaluation_block','event_date','decision_rule'],sort=True):
    pm=g.p_model.to_numpy(float); pk=g.p_market_normalised.to_numpy(float); y=g.Y_event_int.to_numpy(float); wm=float(pm[y==1][0]); wk=float(pk[y==1][0])
    pairedrows.append({'candidate_id':k[0],'probability_variant':k[1],'evaluation_block':k[2],'event_date':k[3],'decision_rule':k[4],'decision_rule_order':int(g.decision_rule_order.iloc[0]),'model_winning_probability':wm,'market_normalised_winning_probability':wk,'model_categorical_log_score':-math.log(min(max(wm,EPS),1.0)),'market_normalised_categorical_log_score':-math.log(min(max(wk,EPS),1.0)),'model_multiclass_brier':float(np.square(pm-y).sum()),'market_normalised_multiclass_brier':float(np.square(pk-y).sum()),'model_minus_market_categorical_log':-math.log(min(max(wm,EPS),1.0))+math.log(min(max(wk,EPS),1.0)),'model_minus_market_multiclass_brier':float(np.square(pm-y).sum()-np.square(pk-y).sum()),'market_book_sum_raw':float(g.market_book_sum_raw.iloc[0])})
paired=pd.DataFrame(pairedrows)
if len(paired)!=924: raise AssertionError('Paired book rows')
pairedsummary=[]
for k,g in paired.groupby(['candidate_id','probability_variant','evaluation_block'],sort=True):
    w=date_weights(g)
    pairedsummary.append({'candidate_id':k[0],'probability_variant':k[1],'evaluation_block':k[2],'exact_common_books':len(g),'exact_common_dates':g.event_date.nunique(),'model_date_balanced_categorical_log_score':float(np.average(g.model_categorical_log_score,weights=w)),'market_date_balanced_normalised_categorical_log_score':float(np.average(g.market_normalised_categorical_log_score,weights=w)),'model_minus_market_categorical_log':float(np.average(g.model_minus_market_categorical_log,weights=w)),'model_date_balanced_multiclass_brier':float(np.average(g.model_multiclass_brier,weights=w)),'market_date_balanced_normalised_multiclass_brier':float(np.average(g.market_normalised_multiclass_brier,weights=w)),'model_minus_market_multiclass_brier':float(np.average(g.model_minus_market_multiclass_brier,weights=w)),'mean_raw_market_book_sum':float(np.average(g.market_book_sum_raw,weights=w))})
paired_summary=pd.DataFrame(pairedsummary)
if len(paired_summary)!=12: raise AssertionError('Paired summary rows')
# Binary exact-common summaries.
binaryrows=[]
for k,g in exact.groupby(['candidate_id','probability_variant','evaluation_block'],sort=True):
    w=date_weights(g)
    binaryrows.append({'candidate_id':k[0],'probability_variant':k[1],'evaluation_block':k[2],'contract_rows':len(g),'dates':g.event_date.nunique(),'model_date_balanced_mean_binary_brier':float(np.average(g.model_binary_brier,weights=w)),'market_raw_date_balanced_mean_binary_brier':float(np.average(g.market_raw_binary_brier,weights=w)),'market_normalised_date_balanced_mean_binary_brier':float(np.average(g.market_normalised_binary_brier,weights=w)),'model_date_balanced_mean_binary_log_score':float(np.average(g.model_binary_log_score,weights=w)),'market_raw_date_balanced_mean_binary_log_score':float(np.average(g.market_raw_binary_log_score,weights=w)),'market_normalised_date_balanced_mean_binary_log_score':float(np.average(g.market_normalised_binary_log_score,weights=w))})
binary_summary=pd.DataFrame(binaryrows)
# Decision-rule summaries.
rulerows=[]
for k,g in paired.groupby(['candidate_id','probability_variant','evaluation_block','decision_rule'],sort=True):
    w=date_weights(g)
    rulerows.append({'candidate_id':k[0],'probability_variant':k[1],'evaluation_block':k[2],'decision_rule':k[3],'books':len(g),'dates':g.event_date.nunique(),'model_categorical_log_score':float(np.average(g.model_categorical_log_score,weights=w)),'market_categorical_log_score':float(np.average(g.market_normalised_categorical_log_score,weights=w)),'model_multiclass_brier':float(np.average(g.model_multiclass_brier,weights=w)),'market_multiclass_brier':float(np.average(g.market_normalised_multiclass_brier,weights=w))})
rule_summary=pd.DataFrame(rulerows)
# Support summary.
support=pd.DataFrame([
{'evaluation_block':'INTERNAL_HOLDOUT','model_only_books_per_candidate_variant':40,'model_only_contract_rows_per_candidate_variant':440,'exact_market_common_books_per_candidate_variant':40,'exact_market_common_contract_rows_per_candidate_variant':440},
{'evaluation_block':'EXTERNAL_TEST','model_only_books_per_candidate_variant':119,'model_only_contract_rows_per_candidate_variant':1309,'exact_market_common_books_per_candidate_variant':114,'exact_market_common_contract_rows_per_candidate_variant':1254},
])
# Calibration effect on model-only support.
effect=[]
for (cid,block),g in model_summary.groupby(['candidate_id','evaluation_block']):
    u=g[g.probability_variant.eq('UNCALIBRATED')].iloc[0]; c=g[g.probability_variant.eq('CALIBRATED')].iloc[0]
    effect.append({'candidate_id':cid,'evaluation_block':block,'calibrated_minus_uncalibrated_categorical_log':float(c.date_balanced_mean_categorical_log_score-u.date_balanced_mean_categorical_log_score),'calibrated_minus_uncalibrated_multiclass_brier':float(c.date_balanced_mean_multiclass_brier-u.date_balanced_mean_multiclass_brier),'calibration_improves_categorical_log':bool(c.date_balanced_mean_categorical_log_score<u.date_balanced_mean_categorical_log_score),'calibration_improves_multiclass_brier':bool(c.date_balanced_mean_multiclass_brier<u.date_balanced_mean_multiclass_brier)})
effect=pd.DataFrame(effect)
# Thesis-ready figures: concise labels, separate holdout/June panels, and restricted axes with actual outlier annotations.
figure_inventory=[]
for block in ['INTERNAL_HOLDOUT','EXTERNAL_TEST']:
    block_title=BLOCK_TITLE[block]
    plot=paired_summary[paired_summary.evaluation_block.eq(block)].copy()
    plot['label']=plot.apply(lambda r:f"{SHORT_NAME[r.candidate_id]} — {SHORT_VARIANT[r.probability_variant]}",axis=1)
    plot=plot.sort_values(['candidate_id','probability_variant']).reset_index(drop=True)
    path=FIG/f"18wD_{'holdout' if block=='INTERNAL_HOLDOUT' else 'external'}_exact_common_categorical_log.png"
    save_horizontal(plot,'label','model_date_balanced_categorical_log_score',f'{block_title}: exact-common categorical log score','Categorical log score',path,xlim=(0,5),clip_annotations=True)
    figure_inventory.append({'figure':path.name,'evaluation_block':block,'metric':'categorical_log','axis_restricted':True})
    path=FIG/f"18wD_{'holdout' if block=='INTERNAL_HOLDOUT' else 'external'}_exact_common_multiclass_brier.png"
    save_horizontal(plot,'label','model_date_balanced_multiclass_brier',f'{block_title}: exact-common multiclass Brier','Multiclass Brier',path,xlim=(0,1.6),clip_annotations=True)
    figure_inventory.append({'figure':path.name,'evaluation_block':block,'metric':'multiclass_brier','axis_restricted':False})
    eff=effect[effect.evaluation_block.eq(block)].copy(); eff['label']=eff.candidate_id.map(SHORT_NAME); eff=eff.sort_values('candidate_id').reset_index(drop=True)
    path=FIG/f"18wD_{'holdout' if block=='INTERNAL_HOLDOUT' else 'external'}_calibration_effect_categorical_log.png"
    save_horizontal(eff,'label','calibrated_minus_uncalibrated_categorical_log',f'{block_title}: calibration effect on categorical log score','Calibrated minus uncalibrated',path,xlim=(-3,0.6),clip_annotations=True,zero_line=True)
    figure_inventory.append({'figure':path.name,'evaluation_block':block,'metric':'calibration_effect_categorical_log','axis_restricted':True})
    reference=[{'label':'Market','score':float(plot.market_date_balanced_normalised_categorical_log_score.iloc[0])}]
    for r in plot.itertuples(index=False): reference.append({'label':f"{SHORT_NAME[r.candidate_id]} — {SHORT_VARIANT[r.probability_variant]}",'score':r.model_date_balanced_categorical_log_score})
    reference=pd.DataFrame(reference)
    path=FIG/f"18wD_{'holdout' if block=='INTERNAL_HOLDOUT' else 'external'}_market_model_categorical_log_comparison.png"
    save_horizontal(reference,'label','score',f'{block_title}: market and model categorical log','Categorical log score',path,xlim=(0,5),clip_annotations=True)
    figure_inventory.append({'figure':path.name,'evaluation_block':block,'metric':'market_model_categorical_log','axis_restricted':True})
figure_inventory=pd.DataFrame(figure_inventory)
# Checks.
checks=pd.DataFrame([
{'check':'pre_unblinding_source_inventories_clean','passed':True,'detail':'18wA-C source inventories contain no full outcome, market or exact-common path','blocking':True},
{'check':'outcome_free_definition_matches_full_panel','passed':True,'detail':'verified after unblinding on all definition columns','blocking':True},
{'check':'blind_release_verified_before_unblinding','passed':True,'detail':'18wC outcome-blind boundary passed','blocking':True},
{'check':'model_outcome_rows_10494','passed':len(model)==10494,'detail':f'rows={len(model)}','blocking':True},
{'check':'model_book_scores_954','passed':len(books)==954,'detail':f'rows={len(books)}','blocking':True},
{'check':'model_summary_rows_12','passed':len(model_summary)==12,'detail':f'rows={len(model_summary)}','blocking':True},
{'check':'exact_common_contract_rows_1694','passed':len(eval_common)==1694,'detail':f'rows={len(eval_common)}','blocking':True},
{'check':'exact_common_books_154','passed':eval_common[['event_date','decision_rule']].drop_duplicates().shape[0]==154,'detail':'40 holdout plus 114 external','blocking':True},
{'check':'exact_joined_model_rows_10164','passed':len(exact)==10164,'detail':f'rows={len(exact)}','blocking':True},
{'check':'paired_book_rows_924','passed':len(paired)==924,'detail':f'rows={len(paired)}','blocking':True},
{'check':'market_normalised_books_sum_to_one','passed':eval_common.groupby(['event_date','decision_rule']).p_market_normalised.sum().apply(lambda v:np.isclose(v,1,atol=1e-12)).all(),'detail':'coherent comparison books','blocking':True},
{'check':'market_not_used_for_fit_or_calibration','passed':True,'detail':'market loaded only after frozen release verification','blocking':True},
{'check':'no_holdout_label_refit','passed':not release.refit_on_holdout_labels.any(),'detail':'identical pre-holdout fit','blocking':True},
])
if not checks.passed.all(): raise AssertionError(checks[~checks.passed].to_string(index=False))
issues=pd.DataFrame(columns=['issue_level','issue_code','candidate_id','probability_variant','evaluation_block','event_date','decision_rule','market_id','detail','blocking'])
outputs={
'18wD_model_probability_outcome_panel.csv':model,
'18wD_model_book_score_panel.csv':books,
'18wD_model_score_summary.csv':model_summary,
'18wD_exact_common_market_model_contract_panel.csv':exact,
'18wD_exact_common_market_model_book_scores.csv':paired,
'18wD_exact_common_score_summary.csv':paired_summary,
'18wD_exact_common_binary_score_summary.csv':binary_summary,
'18wD_exact_common_score_by_rule.csv':rule_summary,
'18wD_evaluation_support_summary.csv':support,
'18wD_calibration_effect_summary.csv':effect,
'18wD_figure_inventory.csv':figure_inventory,
'18wD_integrity_checks.csv':checks,
'18wD_issues.csv':issues,
}
for name,df in outputs.items():
    z=df.copy()
    for col in z.columns:
        if 'date' in col.lower() and pd.api.types.is_datetime64_any_dtype(z[col]): z[col]=z[col].dt.strftime('%Y-%m-%d')
        if 'timestamp' in col.lower() or 'cutoff' in col.lower() or col.lower().endswith('_utc') or 'available' in col.lower() or 'initialisation' in col.lower(): z[col]=z[col].astype('string')
    z.to_csv(OUT/name,index=False)
protocol={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','unblinding_order':'verify 18wA-C manifests, protocols, source inventories, outcome-free definition hash, blind release and frozen calibration before loading 18s outcomes and market','pre_unblinding_source_inventory_boundary_verified':True,'outcome_free_definition_matches_full_outcome_panel':True,'model_only_support':{'internal_holdout_books_per_candidate_variant':40,'external_books_per_candidate_variant':119},'exact_market_common_support':{'internal_holdout_books_per_candidate_variant':40,'external_books_per_candidate_variant':114},'market_categorical_probability':'raw market probabilities normalised within each complete eleven-contract book','market_raw_probabilities_retained':True,'market_used_for_model_fitting':False,'market_used_for_candidate_selection':False,'market_used_for_calibration':False,'refit_on_holdout_labels':False,'trading_evaluation_pending':True}
(OUT/'18wD_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
sources=pd.DataFrame([
{'input_role':'18wA_source_inventory','path':str(WA_SOURCES.relative_to(ROOT)),'rows':len(pd.read_csv(WA_SOURCES)),'sha256':sha(WA_SOURCES)},
{'input_role':'18wB_source_inventory','path':str(WB_SOURCES.relative_to(ROOT)),'rows':len(pd.read_csv(WB_SOURCES)),'sha256':sha(WB_SOURCES)},
{'input_role':'18wC_source_inventory','path':str(WC_SOURCES.relative_to(ROOT)),'rows':len(pd.read_csv(WC_SOURCES)),'sha256':sha(WC_SOURCES)},
{'input_role':'outcome_free_contract_definition_panel','path':str(DEF_PANEL.relative_to(ROOT)),'rows':len(definitions),'sha256':sha(DEF_PANEL)},
{'input_role':'18wC_blind_probability_release','path':str(RELEASE.relative_to(ROOT)),'rows':len(release),'sha256':sha(RELEASE)},
{'input_role':'18wB_selected_calibration_parameters','path':str(PARAM.relative_to(ROOT)),'rows':len(params),'sha256':sha(PARAM)},
{'input_role':'18s_contract_outcomes','path':str(CONTRACT.relative_to(ROOT)),'rows':len(contracts),'sha256':sha(CONTRACT)},
{'input_role':'18s_market_scoring_panel','path':str(MARKET.relative_to(ROOT)),'rows':len(market),'sha256':sha(MARKET)},
{'input_role':'18s_exact_common_support','path':str(COMMON.relative_to(ROOT)),'rows':len(common),'sha256':sha(COMMON)},
]); sources.to_csv(OUT/'18wD_source_inventory.csv',index=False)
summary={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','pre_unblinding_source_inventory_boundary_verified':True,'outcome_free_definition_matches_full_outcome_panel':True,'thesis_ready_figures':8,'selected_candidates':3,'probability_variants':2,'model_outcome_contract_rows':len(model),'model_book_score_rows':len(books),'model_score_summary_rows':len(model_summary),'internal_holdout_model_books_per_candidate_variant':40,'external_model_books_per_candidate_variant':119,'exact_common_contract_rows':len(eval_common),'exact_common_books':154,'exact_common_holdout_books':40,'exact_common_external_books':114,'exact_joined_model_contract_rows':len(exact),'paired_model_market_book_rows':len(paired),'market_used_for_model_fitting':False,'market_used_for_candidate_selection':False,'market_used_for_calibration':False,'refit_on_holdout_labels':False,'trading_evaluation_pending':True,'issue_rows':0,'integrity_checks_passed':int(checks.passed.sum()),'integrity_checks_total':len(checks)}
(OUT/'18wD_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
(OUT/'18wD_environment.json').write_text(json.dumps({'generated_at_utc':datetime.now(UTC).isoformat(),'python':sys.version,'platform':platform.platform(),'pandas':pd.__version__,'numpy':np.__version__,'matplotlib':plt.matplotlib.__version__,'revision':'v2'},indent=2),encoding='utf-8')
lines=['# 18wD locked holdout and June probability evaluation','','**PASS**','','## Model-only support','','| Block | Books per candidate and variant |','|---|---:|','| Internal holdout | 40 |','| June external test | 119 |','','## Exact market-common support','','| Block | Books | Contract rows |','|---|---:|---:|','| Internal holdout | 40 | 440 |','| June external test | 114 | 1,254 |','','## Exact-common categorical scores','','| Candidate | Variant | Block | Model log | Market log | Model minus market | Model Brier | Market Brier |','|---|---|---|---:|---:|---:|---:|---:|']
for r in paired_summary.itertuples(index=False): lines.append(f'| {r.candidate_id} | {r.probability_variant} | {r.evaluation_block} | {r.model_date_balanced_categorical_log_score:.6f} | {r.market_date_balanced_normalised_categorical_log_score:.6f} | {r.model_minus_market_categorical_log:.6f} | {r.model_date_balanced_multiclass_brier:.6f} | {r.market_date_balanced_normalised_multiclass_brier:.6f} |')
lines += ['','## Figures','','Eight thesis-ready figures use concise labels and separate internal-holdout and June panels. Restricted categorical-log axes retain annotated actual values for CatBoost outliers.','','The mechanically outcome-blind 18wA-C chain and its source inventories were verified before outcomes or market information were loaded. Trading remains deferred.']
(REPORT/'18wD_holdout_external_probability_evaluation_report.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
manifest=[]
for root in [OUT,REPORT]:
    for p in sorted(root.rglob('*')):
        if p.is_file() and p.name!='18wD_sha256_manifest.csv': manifest.append({'path':str(p.relative_to(ROOT)),'size_bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(manifest).to_csv(OUT/'18wD_sha256_manifest.csv',index=False)
print(json.dumps(summary,indent=2)); display(model_summary); display(paired_summary); print('18wD holdout/external probability evaluation: PASS')

Outcome-blind A-C chain and frozen calibration verified before unblinding: PASS


{
  "step": "18wD",
  "generated_at_utc": "2026-07-22T12:27:11.059782+00:00",
  "verdict": "PASS",
  "pre_unblinding_source_inventory_boundary_verified": true,
  "outcome_free_definition_matches_full_outcome_panel": true,
  "thesis_ready_figures": 8,
  "selected_candidates": 3,
  "probability_variants": 2,
  "model_outcome_contract_rows": 10494,
  "model_book_score_rows": 954,
  "model_score_summary_rows": 12,
  "internal_holdout_model_books_per_candidate_variant": 40,
  "external_model_books_per_candidate_variant": 119,
  "exact_common_contract_rows": 1694,
  "exact_common_books": 154,
  "exact_common_holdout_books": 40,
  "exact_common_external_books": 114,
  "exact_joined_model_contract_rows": 10164,
  "paired_model_market_book_rows": 924,
  "market_used_for_model_fitting": false,
  "market_used_for_candidate_selection": false,
  "market_used_for_calibration": false,
  "refit_on_holdout_labels": false,
  "trading_evaluation_pending": true,
  "issue_rows": 0,
  "integrity_checks_pass

,candidate_id,probability_variant,evaluation_block,books,dates,date_balanced_mean_categorical_log_score,date_balanced_mean_multiclass_brier,date_balanced_mean_winning_probability,zero_winning_probability_books
0,catboost_quantile_pooled,CALIBRATED,EXTERNAL_TEST,119,30,3.200493,1.079489,0.106490,0
1,catboost_quantile_pooled,CALIBRATED,INTERNAL_HOLDOUT,40,10,3.548893,1.081541,0.092500,0
2,catboost_quantile_pooled,UNCALIBRATED,EXTERNAL_TEST,119,30,21.837264,1.442254,0.063356,93
3,catboost_quantile_pooled,UNCALIBRATED,INTERNAL_HOLDOUT,40,10,24.289809,1.403204,0.057828,35
4,gp_matern32_rule,CALIBRATED,EXTERNAL_TEST,119,30,1.591468,0.746733,0.221773,0
5,gp_matern32_rule,CALIBRATED,INTERNAL_HOLDOUT,40,10,1.240250,0.616519,0.329545,0
6,gp_matern32_rule,UNCALIBRATED,EXTERNAL_TEST,119,30,1.857855,0.723985,0.273148,2
7,gp_matern32_rule,UNCALIBRATED,INTERNAL_HOLDOUT,40,10,1.063872,0.575212,0.383333,0
8,pooled_empirical_residual,CALIBRATED,EXTERNAL_TEST,119,30,1.688120,0.725092,0.250337,1
9,pooled_empirical_residual,CALIBRATED,INTERNAL_HOLDOUT,40,10,1.133471,0.583328,0.360101,0


,candidate_id,probability_variant,evaluation_block,exact_common_books,exact_common_dates,model_date_balanced_categorical_log_score,market_date_balanced_normalised_categorical_log_score,model_minus_market_categorical_log,model_date_balanced_multiclass_brier,market_date_balanced_normalised_multiclass_brier,model_minus_market_multiclass_brier,mean_raw_market_book_sum
0,catboost_quantile_pooled,CALIBRATED,EXTERNAL_TEST,114,30,3.203871,1.260923,1.942949,1.081487,0.648471,0.433017,1.031639
1,catboost_quantile_pooled,CALIBRATED,INTERNAL_HOLDOUT,40,10,3.548893,1.091350,2.457544,1.081541,0.585087,0.496455,1.039287
2,catboost_quantile_pooled,UNCALIBRATED,EXTERNAL_TEST,114,30,21.846419,1.260923,20.585496,1.449981,0.648471,0.801511,1.031639
3,catboost_quantile_pooled,UNCALIBRATED,INTERNAL_HOLDOUT,40,10,24.289809,1.091350,23.198459,1.403204,0.585087,0.818117,1.039287
4,gp_matern32_rule,CALIBRATED,EXTERNAL_TEST,114,30,1.615497,1.260923,0.354574,0.750062,0.648471,0.101591,1.031639
5,gp_matern32_rule,CALIBRATED,INTERNAL_HOLDOUT,40,10,1.240250,1.091350,0.148900,0.616519,0.585087,0.031432,1.039287
6,gp_matern32_rule,UNCALIBRATED,EXTERNAL_TEST,114,30,2.076144,1.260923,0.815221,0.728580,0.648471,0.080109,1.031639
7,gp_matern32_rule,UNCALIBRATED,INTERNAL_HOLDOUT,40,10,1.063872,1.091350,-0.027477,0.575212,0.585087,-0.009875,1.039287
8,pooled_empirical_residual,CALIBRATED,EXTERNAL_TEST,114,30,1.893650,1.260923,0.632727,0.726371,0.648471,0.077901,1.031639
9,pooled_empirical_residual,CALIBRATED,INTERNAL_HOLDOUT,40,10,1.133471,1.091350,0.042122,0.583328,0.585087,-0.001758,1.039287


18wD holdout/external probability evaluation: PASS
